In [1]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display

from som_rl.som.som_backend import SharpZSOMBackend, SharpZBackendConfig
from som_rl.eval.ideal_som import get_perfect_bins

DATA = "data"  # adjust if needed

In [2]:
# Load SOM / correlation data and create backend

npz = np.load(f"{DATA}/correlations_dl/npairs_compact.npz")
dd, dr, rd, rr = npz["dd"], npz["dr"], npz["rd"], npz["rr"]

node_indices_photo = np.load(f"{DATA}/node_indices_fc.npy")
initial_node_bins  = np.load(f"{DATA}/nodes_labels_initial_fc_200_ext.npy")
reference_labels   = np.load(f"{DATA}/reference_labels_fc_fc.npy")
weighted_clusters  = np.load(f"{DATA}/weighted_clusters_5c_1t200.npy")
umatrix = np.load(f"{DATA}/umat.npy")

perfect_bins = get_perfect_bins()

b_cfg = SharpZBackendConfig(
    grid_shape=(200, 200),
    num_bins=11,
    objective_mode="energy",
    proposal_bin_mode="uniform_other",
)


def make_backend():
    return SharpZSOMBackend(
        cfg=b_cfg,
        dd=dd, dr=dr, rd=rd, rr=rr,
        node_indices_photo=node_indices_photo,
        initial_node_bins=initial_node_bins,
        reference_labels=reference_labels,
        umatrix=umatrix,
        weighted_clusters=weighted_clusters,
    )

backend = make_backend()
K = backend.K
H, W = backend.H, backend.W
L = backend.levels_available()
print(f"Backend ready: grid {H}x{W}, K={K}, levels={L}")

Backend ready: grid 200x200, K=11, levels=201


In [3]:
def compute_deltaE_per_bin(backend, node, level, perfect_bins):
    """
    For the given node (and optional level for cluster), compute ΔE if we set
    that node/cluster to each bin. Returns dE per bin, current bin, best bin,
    true bin (perfect_bins-based), nodes (for bbox), and is_cluster.
    """
    be = backend
    if level is None:
        level_used = None
    elif be.weighted_clusters is not None and 0 <= int(level) < be.levels_available():
        level_used = int(level)
    else:
        level_used = None
    nodes, lvl, cid = be._nodes_for_action_target(node=node, level=level_used)
    if nodes.size == 0:
        return None

    current_bin = int(be._cluster_current_bin_weighted_mode(nodes))
    # True bin: per-node perfect_bins[node]; for cluster use weighted mode over nodes
    w = be.node_gal_counts[nodes].astype(np.float64)
    true_bins = perfect_bins[nodes].astype(np.int32)
    counts = np.bincount(true_bins, weights=w, minlength=be.K)
    true_bin = int(np.argmax(counts))

    dE_per_bin = np.full(be.K, np.nan, dtype=np.float64)
    for b in range(be.K):
        prop = be.make_proposal_set_bin(new_bin=b, node=node, level=level_used)
        if prop.get("type") == "noop":
            continue
        ev = be.evaluate_proposal_deltaE(prop)
        dE_per_bin[b] = float(ev["dE"])
        be.revert(ev["undo"])

    # Current bin: no move → dE = 0
    dE_per_bin[current_bin] = 0.0

    valid = np.isfinite(dE_per_bin)
    best_bin = int(np.argmax(dE_per_bin)) if np.any(valid) else current_bin

    return {
        "dE_per_bin": dE_per_bin,
        "current_bin": current_bin,
        "best_bin": best_bin,
        "true_bin": true_bin,
        "nodes": nodes,
        "is_cluster": lvl is not None,
    }


def cluster_bbox(backend, nodes):
    """(min_x, max_x, min_y, max_y) in grid coordinates for drawing a box."""
    nodes = np.asarray(nodes, dtype=np.int32)
    if nodes.size == 0:
        return None
    xs, ys = [], []
    for idx in nodes:
        x, y = backend.index_to_xy(int(idx))
        xs.append(x)
        ys.append(y)
    return min(xs), max(xs), min(ys), max(ys)

In [4]:
# Interactive figure: SOM on top, bar chart below; level slider + click on SOM

def run_inspector(backend, perfect_bins):
    be = backend
    K, H, W = be.K, be.H, be.W
    L = be.levels_available()
    level_max = max(0, L - 1)

    plt.ioff()  # avoid figure auto-showing (we display it once in VBox)
    fig, (ax_som, ax_bar) = plt.subplots(2, 1, figsize=(9, 7), gridspec_kw={"height_ratios": [2.2, 0.6]})
    plt.subplots_adjust(hspace=0.2)

    # Level slider
    level_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=level_max,
        step=1,
        description="Level:",
        continuous_update=False,
        layout=widgets.Layout(width="400px"),
    )

    cluster_rect = None  # persistent patch for red cluster box

    def draw_som(ax, highlight_mask=None):
        ax.clear()
        # Show current bins grid (actual state)
        grid = be.bins_grid()
        # discrete colormap with one colour per bin; use half-offset so ticks sit at bin centres
        cmap = plt.get_cmap("viridis", K)
        im = ax.imshow(grid, vmin=-0.5, vmax=K - 0.5, cmap=cmap, aspect="equal", origin="upper",
                   extent=(-0.5, W - 0.5, H - 0.5, -0.5))
        # If we have a cluster mask, outline its exact shape
        if highlight_mask is not None:
            ax.contour(np.arange(W), np.arange(H), highlight_mask.astype(float), levels=[0.5], colors="red", linewidths=2)
        ax.set_xlim(-0.5, W - 0.5)
        ax.set_ylim(H - 0.5, -0.5)
        ax.set_aspect("equal", adjustable="box")
        #ax.set_xlabel("x")
        #ax.set_ylabel("y")
        ax.set_title("SOM bins (click a node)")
        return im

    def draw_bar(ax, result):
        ax.clear()
        # Match bar plot width to SOM (narrower)
        som_pos = ax_som.get_position()
        bar_pos = ax.get_position()
        ax.set_position([som_pos.x0, bar_pos.y0, som_pos.width, bar_pos.height])
        dE = result["dE_per_bin"]
        cur = result["current_bin"]
        true_bin = result["true_bin"]  # perfect SOM bin for this cluster
        bins = np.arange(K)

        # Column highlights: current=blue; perfect SOM bin=yellow; green if same
        if cur == true_bin:
            ax.axvspan(cur - 0.5, cur + 0.5, color="#c7e9c0", alpha=0.6, zorder=0)  # green
        else:
            ax.axvspan(cur - 0.5, cur + 0.5, color="#c6dbef", alpha=0.6, zorder=0)   # current = blue
            ax.axvspan(true_bin - 0.5, true_bin + 0.5, color="#f7dc6f", alpha=0.6, zorder=0)  # perfect SOM = yellow

        fill = np.where(np.isfinite(dE), dE, 0.0)
        ax.bar(bins, fill, color="#444444", edgecolor="black", linewidth=0.5, zorder=1)
        ax.axhline(0, color="gray", linestyle="-", linewidth=0.5)
        ax.set_xlabel("Bin")
        ax.set_ylabel("ΔE")
        ax.set_title("ΔE if node/cluster set to each bin ")
        ax.set_xticks(bins)
        ax.set_xticklabels([str(b) for b in bins])
        ax.set_xlim(-0.5, K - 0.5)
        # Auto-scale y to fit the bars
        valid_vals = fill[np.isfinite(dE)]
        if len(valid_vals) > 0:
            y_lo, y_hi = np.min(valid_vals), np.max(valid_vals)
            pad = max((y_hi - y_lo) * 0.08, 1e-10) if y_hi > y_lo else 0.1
            ax.set_ylim(min(y_lo - pad, 0), max(y_hi + pad, 0))
        else:
            ax.set_ylim(-0.1, 0.1)
        ax.grid(axis="y", alpha=0.3)

    # Initial SOM (no selection) + segmented colourbar for bins
    im = draw_som(ax_som, None)
    cbar = fig.colorbar(im, ax=ax_som, fraction=0.046, pad=0.04)
    cbar.set_ticks(np.arange(K))
    cbar.set_ticklabels([str(b) for b in range(K)])
    cbar.set_label("Bin", size=10)
    cbar.ax.tick_params(labelsize=8)

    som_pos = ax_som.get_position()
    bar_pos = ax_bar.get_position()
    ax_bar.set_position([som_pos.x0, bar_pos.y0, som_pos.width, bar_pos.height])
    ax_bar.text(0.5, 0.5, "Click a node on the SOM above", transform=ax_bar.transAxes,
                ha="center", va="center", fontsize=12)
    ax_bar.set_xlim(-0.5, K - 0.5)
    ax_bar.set_ylim(0, 1)

    def on_click(event):
        if event.inaxes != ax_som:
            return
        # Convert from display coords to data coords so clicks align with aspect="equal"
        try:
            xdata, ydata = ax_som.transData.inverted().transform((event.x, event.y))
        except Exception:
            return
        ix = int(round(xdata))
        iy = int(round(ydata))
        if not (0 <= ix < W and 0 <= iy < H):
            return
        node = be.xy_to_index(ix, iy)
        level_index = level_slider.value
        level_used = level_index if L > 0 else None
        result = compute_deltaE_per_bin(be, node, level_used, perfect_bins)
        if result is None:
            return
        highlight_mask = None
        if result["is_cluster"] and result["nodes"].size > 0:
            highlight_mask = np.zeros((H, W), dtype=bool)
            for idx in result["nodes"]:
                x, y = be.index_to_xy(int(idx))
                if 0 <= x < W and 0 <= y < H:
                    highlight_mask[y, x] = True
        draw_som(ax_som, highlight_mask)
        draw_bar(ax_bar, result)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", on_click)

    def on_level_change(change):
        # Redraw SOM only (no new selection); keep bar if we had one
        ax_som.clear()
        draw_som(ax_som, None)
        fig.canvas.draw_idle()

    level_slider.observe(on_level_change, names="value")

    # Single output: slider above figure (left-aligned to reduce surrounding whitespace)
    container = widgets.VBox(
        [level_slider, fig.canvas],
        layout=widgets.Layout(align_items="flex-start")
    )
    display(container)


run_inspector(backend, perfect_bins)

In [6]:
import matplotlib.pyplot as plt

fig = plt.gcf()  # current figure (the inspector)
fig.savefig("plots/degens/big_failure.pdf", bbox_inches="tight")